In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df2 = pd.read_csv(
    r"C:\Users\workk\OneDrive\Desktop\churn prediction project\WA_Fn-UseC_-Telco-Customer-Churn (1).csv"
)

In [3]:
df2.drop(
    "customerID",
    axis=1,
    inplace=True
)

In [4]:
df2["TotalCharges"] = pd.to_numeric(
    df2["TotalCharges"],
    errors='coerce'
)

In [5]:
df2.dropna(inplace=True)

In [6]:
df2["Charges_Per_Tenure"] = (
    df2["TotalCharges"] /
    (df2["tenure"] + 1)
)

In [7]:
df2["HighValueCustomer"] = (
    df2["MonthlyCharges"] > 80
).astype(int)

In [8]:
df2["LongTermCustomer"] = (
    df2["tenure"] > 24
).astype(int)

In [9]:
df2["Churn"] = df2["Churn"].map({
    "Yes": 1,
    "No": 0
})

In [10]:
df2 = pd.get_dummies(
    df2,
    drop_first=True
)

In [11]:
X = df2.drop("Churn", axis=1)

y = df2["Churn"]

In [12]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [13]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(
    X_train
)

X_test_scaled = scaler.transform(
    X_test
)

In [14]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(
    X_train
)

X_test_scaled = scaler.transform(
    X_test
)

In [15]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(
    X_train
)

X_test_scaled = scaler.transform(
    X_test
)

In [18]:
from imblearn.over_sampling import SMOTE

In [19]:
smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train_scaled,
    y_train
)

In [20]:
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier
)

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    classification_report
)

In [21]:
models = {

    "RandomForest":
        RandomForestClassifier(
            n_estimators=300,
            random_state=42
        ),

    "ExtraTrees":
        ExtraTreesClassifier(
            n_estimators=300,
            random_state=42
        ),

    "GradientBoost":
        GradientBoostingClassifier(
            n_estimators=200,
            learning_rate=0.05,
            random_state=42
        ),

    "AdaBoost":
        AdaBoostClassifier(
            n_estimators=200,
            learning_rate=0.05,
            random_state=42
        ),

    "XGBoost":
        XGBClassifier(
            n_estimators=500,
            learning_rate=0.03,
            max_depth=5,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric='logloss',
            random_state=42
        ),

    "LightGBM":
        LGBMClassifier(
            n_estimators=500,
            learning_rate=0.03,
            random_state=42
        ),

    "CatBoost":
        CatBoostClassifier(
            iterations=500,
            learning_rate=0.03,
            depth=5,
            verbose=0,
            random_state=42
        )
}

In [22]:
results = []

for name, model in models.items():

    print(f"Training {name}...")

    model.fit(
        X_train_smote,
        y_train_smote
    )

    preds = model.predict(
        X_test_scaled
    )

    probs = model.predict_proba(
        X_test_scaled
    )[:,1]

    accuracy = accuracy_score(
        y_test,
        preds
    )

    roc_auc = roc_auc_score(
        y_test,
        probs
    )

    results.append({
        "Model": name,
        "Accuracy": accuracy,
        "ROC_AUC": roc_auc
    })

    print(f"{name} completed")

Training RandomForest...
RandomForest completed
Training ExtraTrees...
ExtraTrees completed
Training GradientBoost...
GradientBoost completed
Training AdaBoost...
AdaBoost completed
Training XGBoost...
XGBoost completed
Training LightGBM...
[LightGBM] [Info] Number of positive: 4130, number of negative: 4130
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001910 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2640
[LightGBM] [Info] Number of data points in the train set: 8260, number of used features: 33
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000


C:\Users\workk\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\workk\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LightGBM completed
Training CatBoost...
CatBoost completed


In [23]:
results_df = pd.DataFrame(results)

results_df.sort_values(
    by="ROC_AUC",
    ascending=False
)

,Model,Accuracy,ROC_AUC
2,GradientBoost,0.774698,0.834824
6,CatBoost,0.780384,0.833985
4,XGBoost,0.778252,0.828315
3,AdaBoost,0.675906,0.828138
5,LightGBM,0.786070,0.826981
0,RandomForest,0.778962,0.819735
1,ExtraTrees,0.773276,0.798174


In [1]:
from catboost import CatBoostClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    roc_auc_score,
    confusion_matrix
)

In [2]:
final_model = CatBoostClassifier(

    iterations=1000,
    learning_rate=0.02,
    depth=6,

    loss_function='Logloss',

    eval_metric='AUC',

    random_seed=42,

    verbose=0
)

In [7]:
print("Checking variables...\n")

variables = [

    "X_train_smote",
    "y_train_smote",
    "X_test_scaled",
    "y_test",
    "X"

]

for var in variables:

    if var in globals():

        print(f"{var} EXISTS")

    else:

        print(f"{var} NOT FOUND")

Checking variables...

X_train_smote NOT FOUND
y_train_smote NOT FOUND
X_test_scaled NOT FOUND
y_test NOT FOUND
X NOT FOUND


In [ ]:


# -------------------------
# SMOTE
# -------------------------

smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(

    X_train_scaled,

    y_train
)

print("ALL VARIABLES CREATED SUCCESSFULLY")